Function that will be used to crawl the publication sites and acquire a list of unique publications along with other info

In [89]:
import requests
from bs4 import BeautifulSoup as bsSoup
import time
import sys
import pandas as pd
import warnings
import os

In [90]:
# scrape the links for all the publications of https://sitescrape.awh.durham.ac.uk/comp42315/publicationfull_year_characteranimation.htm
publicationsIndex = "https://sitescrape.awh.durham.ac.uk/comp42315/publicationfull_year_characteranimation.htm"

def getLinks (url: str) -> list :
    sitePrefix = "https://sitescrape.awh.durham.ac.uk/comp42315/"

    if (not isinstance(url, str)) :
        print("The URL needs to be a string!")
        return []

    # wait a little so we don't overload the server
    time.sleep(2)
    publicationsIndex = requests.get(url, verify = False)

    if (publicationsIndex.status_code != 200) :
        print (f"Error; status code returned: {publicationsIndex.status_code}")
        return []

    publicationsIndexSoup = bsSoup(publicationsIndex.content, "html.parser").body

    if (publicationsIndexSoup == None) :
        print ("Nothin to parse on the site")
        return []

    publicationLinksA = publicationsIndexSoup.find("p", class_ = "TextOption").find_all("a")

    if (publicationLinksA == None) :
        print ("No links found")
        return []
    
    links = [sitePrefix + n.get("href") for n in publicationLinksA]

    if (links[0] == None) :
        print("No links found")
        return []

    return links

publicationLinks = [publicationsIndex] + getLinks(publicationsIndex)

def scrapeImpactScores (urls: list) -> dict :
    impactScores = {}
    # if a site has been seen, ignore it
    seenPages = set()
    n = 0
    
    for url in urls :
        #n += 1
        #print(n)
        uniqueScoresOnThePage = []
        # change that 0 to 1 later
        time.sleep(0)
        url = url.replace("year", "impactfactor")
        
        scorePage = requests.get(url, verify = False)

        if (scorePage.status_code != 200) :
            print(f"Failed for link {url}, status code: {scorePage.status_code}, continuing execution for the remaining links")
            continue

        scorePageSoup = bsSoup(scorePage.content, "html.parser").body

        if (scorePageSoup == None) :
            print (f"Found nothing to parse on site {url}, continuing execution for the remaining links")
            continue

        scorePageSoupDiv = scorePageSoup.find("div", id = "divBackground")
        
        if (scorePageSoupDiv == None) :
            print (f"Couldn't find scores on the page {url}, continuing execution for the remaining links")
            continue

        scorePageSoupP = scorePageSoupDiv.find_all("p", class_ = "TextOption")

        if (scorePageSoupP == None) :
            print (f"Couldn't find scores on the page {url}, continuing execution for the remaining links")
            continue

        paragraphWithScores = scorePageSoupP [2]

        uniqueScoresOnThePage = [n.text for n in paragraphWithScores.find_all("a")]

        if (len(uniqueScoresOnThePage) == 0) :
            print (f"Couldn't find scores on the page {url}, continuing execution for the remaining links")
            continue

        # for each link in unique links find the publication titles that correspond
        for score in uniqueScoresOnThePage :
            currentH2 = scorePageSoup.find("h2", id = score)

            if (currentH2 == None) :
                # no publications under this score
                continue

            if (score not in impactScores) :
                impactScores [score] = []

            div = currentH2.findNext()

            publicationsWithThisScore = div.find_all("div", class_ = "w3-cell-row")

            if (publicationsWithThisScore == None) :
                # no publications under this score
                continue

            for publication in publicationsWithThisScore :
                publicationText = publication.find("div", class_ = "w3-container w3-cell w3-mobile w3-cell-middle")

                # I'd need to find the title here
                title = publicationText.text.split("by")[0].strip()

                if (title in seenPages) :
                    continue

                seenPages.add(title)
                impactScores[score].append(publicationText.text)
                #print(title)

    return impactScores
                                                      
#print (publicationLinks)
#print(scrapePublications(publicationLinks))   
scoreDict = scrapeImpactScores(publicationLinks)
print(scoreDict)

c:\Users\Cobra Kai\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'sitescrape.awh.durham.ac.uk'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\Cobra Kai\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'sitescrape.awh.durham.ac.uk'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\Cobra Kai\AppData\Local\Temp\ipykernel_8348\3867983403.py:98: DeprecationWarning: Call to deprecated method findNext. (Replaced by find_next) -- Deprecated since version 4.0.0.
  div = currentH2.findNext()
c:\Users\Cobra Kai\AppData\Local\Programs\Python\Python314\Lib\si

{'5.0+': ['\t\r\n\t\tInteraction Patches for Multi-Character Animation by Hubert P. H. Shum, Taku Komura and Shu Takagi in 2010ACM Transactions on Graphics (TOG) - Proceedings of the 2008 ACM SIGGRAPH Asia (SIGGRAPH Asia)\n Webpage\n', '\t\r\n\t\tA Unified Deep Metric Representation for Mesh Saliency Detection and Non-rigid Shape Matching by Brian K. S. Isaac-Medina, Matthew Poyser, Daniel Organisciak, Chris G. Willcocks, Toby P. Breckon and Hubert P. H. Shum in 2020IEEE Transactions on Multimedia (TMM)\n Webpage\n', '\t\r\n\t\tSparse Metric-based Mesh Saliency by John Hartley, Hubert P. H. Shum, Edmond S. L. Ho, He Wang and Subramanian Ramamoorthy in 2021Neurocomputing\n Webpage\n', '\t\r\n\t\tLMZMPM: Local Modified Zernike Moment Per-unit Mass for Robust Human Face Recognition by Luca Crosato, Chongfeng Wei, Edmond S. L. Ho and Hubert P. H. Shum in 2021IEEE Transactions on Information Forensics and Security (TIFS)\n Webpage\n', '\t\r\n\t\tTwo-Stage Human Verification using HandCAPTCH

c:\Users\Cobra Kai\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'sitescrape.awh.durham.ac.uk'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\Cobra Kai\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'sitescrape.awh.durham.ac.uk'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [91]:
def scrapeAdditionalInformation (urls: list, replace : str) -> dict :
    scrapedData = {}
    seenPages = set()

    for url in urls :
        uniqueValuesOnPage = []
        time.sleep(0)
        url = url.replace("year", replace)

        page = requests.get(url, verify = False)

        if (page.status_code != 200) :
            print(f"Failed for link {url}, status code: {page.status_code}, continuing execution for the remaining links")
            continue

        pageSoup = bsSoup(page.content, "html.parser").body

        if (pageSoup == None) :
            print (f"Found nothing to parse on site {url}, continuing execution for the remaining links")
            continue

        pageSoupDiv = pageSoup.find("div", id = "divBackground")

        if (pageSoupDiv == None) :
            print (f"Couldn't find {replace} on the page {url}, continuing execution for the remaining links")
            continue

        pageSoupP = pageSoupDiv.find_all("p", class_ = "TextOption")

        if (pageSoupP == None) :
            print (f"Couldn't find {replace} on the page {url}, continuing execution for the remaining links")
            continue

        paragraphWithInfo = pageSoupP [2]

        valueTags = paragraphWithInfo.find_all("a")

        uniqueValuesOnPage = [n.text for n in valueTags]

        if (len(uniqueValuesOnPage) == 0) :
            print (f"Couldn't find {replace} on the page {url}, continuing execution for the remaining links")
            continue

        for value in uniqueValuesOnPage :
            currentH2 = pageSoup.find("h2", id = value)

            if (currentH2 == None) :
                continue

            if (value not in scrapedData) :
                scrapedData [value] = []

            div = currentH2.findNext()

            publicationsWithThisInfo = div.find_all("div", class_ = "w3-cell-row")

            if (publicationsWithThisInfo == None) :
                continue

            for publication in publicationsWithThisInfo :
                publicationText = publication.find("div", class_ = "w3-container w3-cell w3-mobile w3-cell-middle")

                title = publicationText.text.split("by")[0].strip()

                if (title in seenPages) :
                    continue

                seenPages.add(title)
                scrapedData[value].append(title)

    return scrapedData

typeDict = scrapeAdditionalInformation(publicationLinks, "type")
print (typeDict)
print(typeDict.keys())

citationsDict = scrapeAdditionalInformation(publicationLinks, "citation")
print (citationsDict)
print (citationsDict.keys())

c:\Users\Cobra Kai\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'sitescrape.awh.durham.ac.uk'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\Cobra Kai\AppData\Local\Temp\ipykernel_8348\3195796701.py:53: DeprecationWarning: Call to deprecated method findNext. (Replaced by find_next) -- Deprecated since version 4.0.0.
  div = currentH2.findNext()
c:\Users\Cobra Kai\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'sitescrape.awh.durham.ac.uk'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\Cobra Kai\AppData\Local\Programs\Python\Python314\Lib\si

{'Journal Papers': ['Spatio-temporal Manifold Learning for Human Motions via Long-horizon Modeling', 'A Quadruple Diffusion Convolutional Recurrent Network for Human Motion Prediction', 'GAN-based Reactive Motion Synthesis with Class-aware Discriminators for Human-human Interaction', 'A Generic Framework for Editing and Synthesizing Multimodal Data with Relative Emotion Strength', 'Multi-layer Lattice Model for Real-Time Dynamic Character Deformation', 'Human Motion Variation Synthesis with Multivariate Gaussian Processes', 'Topology Aware Data-Driven Inverse Kinematics', 'Natural Preparation Behavior Synthesis', 'Simulating Multiple Character Interactions with Collaborative and Adversarial Goals', 'Angular Momentum Guided Motion Concatenation', 'Interaction Patches for Multi-Character Animation', 'Interaction-based Human Activity Comparison', 'Abnormal Infant Movements Classification with Deep Learning on Pose-based Features', 'Automatic Musculoskeletal and Neurological Disorder Diagn

c:\Users\Cobra Kai\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'sitescrape.awh.durham.ac.uk'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\Cobra Kai\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'sitescrape.awh.durham.ac.uk'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\Cobra Kai\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'sitescrape.awh.durham.ac.uk'. Adding certificate verification is strongly advised. See: https://url

{'100+': ['A Quadruple Diffusion Convolutional Recurrent Network for Human Motion Prediction', 'Interaction Patches for Multi-Character Animation', 'Simulating Multiple Character Interactions with Collaborative and Adversarial Goals', 'Simulating Competitive Interactions using Singly Captured Motions', 'Simulating Interactions of Avatars in High Dimensional State Space', 'Real-Time Posture Reconstruction for Microsoft Kinect'], '50+': ['Real-time Physical Modelling of Character Movements with Microsoft Kinect', 'Interaction-based Human Activity Comparison', 'Interpreting Deep Learning based Cerebral Palsy Prediction with Channel Attention', 'A Two-Stream Recurrent Network for Skeleton-Based Human Interaction Recognition', 'Kinect Posture Reconstruction based on a Local Mixture of Gaussian Process Models', 'Posture Reconstruction Using Kinect with a Probabilistic Model', 'Bi-projection based Foreground-aware Omnidirectional Depth Prediction', 'Filtered Pose Graph for Efficient Kinect Po

c:\Users\Cobra Kai\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'sitescrape.awh.durham.ac.uk'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


This information should then be stored and displayed in an appropriate format. Displayed to the user should be: publication title, publication venue, year and author list of every unique publication record on the target website.

In [92]:
def dictConverter (dataDict : dict) -> dict :
    convertedDict = {}

    for k in dataDict :
        for n in dataDict[k] :
            convertedDict[n] = k

    return convertedDict

typeDictClean = dictConverter(typeDict)
citationsDictClean = dictConverter(citationsDict)


In [ ]:
# extract info from score dict
# create a new dictionary with cleaned data title: [[authors], year, publication venue]
publications = {}
authorsListConcat : str = []
publicationsData : pd.DataFrame
s = 0

for k in scoreDict :
    for n in scoreDict[k]:
        # 110 (started indexing at 0) unique publications it seems
        remainder = ""
        initialClean = n.translate({ord(i): None for i in '\t\r\n'})
        # remove the "Webpage" at the end
        initialClean_2 = initialClean[:-8]

        # ok, it's important that we only get the first occurance of by
        splitAt = initialClean_2.find("by")
        title = initialClean_2[:splitAt].strip()
        remainder = initialClean_2[splitAt + 3:]

        # find that title in typeDictClean and citationsDictClean
        typeOfPublication = typeDictClean[title]
        noOfCitations = citationsDictClean[title]

        splitAt = remainder.find(" in ")
        authors = remainder[:splitAt]
        year = remainder[splitAt + 4:splitAt + 8]
        publicationVenue = remainder[splitAt + 8:]

        authorsListConcat.append(authors)

        # splitting authors into a list of names
        authorList1 = authors.split(",")
        authorList2 = authorList1[-1].split("and")
        #print("---")
        authorList1Strip = [n.strip() for n in authorList1]
        authorList2Strip = [n.strip() for n in authorList2]
        authorList = authorList1Strip[:-1] + authorList2Strip

        # add to dictionary; k for score
        publications[f"{title}"] = [authorList, year, publicationVenue, k, typeOfPublication, noOfCitations]

# also create a pandas dataframe from it
_data = {"title" : [k for k in publications], 
         "authors" : [n for n in authorsListConcat], 
         "year" : [publications[k][1] for k in publications], 
         "publication venue" : [publications[k][2] for k in publications],
         "score" : [publications[k][3] for k in publications],
         "type" : [publications[k][4] for k in publications],
         "number of citations" : [publications[k][5] for k in publications]}

# print (len(_data["publication venue"]))
#print (_data["authors"])

publicationsData = pd.DataFrame(data = _data)

                                                 title  \
0    Interaction Patches for Multi-Character Animation   
1    A Unified Deep Metric Representation for Mesh ...   
2                    Sparse Metric-based Mesh Saliency   
3    LMZMPM: Local Modified Zernike Moment Per-unit...   
4    Two-Stage Human Verification using HandCAPTCHA...   
5    Multiview Discriminative Marginal Metric Learn...   
6    Facial Reshaping Operator for Controllable Fac...   
7    Differential Evolution Algorithm as a Tool for...   
8    A Quadruple Diffusion Convolutional Recurrent ...   
9    Spatio-temporal Manifold Learning for Human Mo...   
10   Simulating Multiple Character Interactions wit...   
11         Interaction-based Human Activity Comparison   
12   Kinect Posture Reconstruction based on a Local...   
13   Interactive Formation Control in Complex Envir...   
14   Multi-layer Lattice Model for Real-Time Dynami...   
15       Topology Aware Data-Driven Inverse Kinematics   
16   Inverse D

In [ ]:
# print publications dictionary to the console
def printScrapedDataToConsole () -> None :
    for k in publications :
        print ("---")
        print (f"Title: {k}")
        sys.stdout.write("Authors: ")

        count = 0
        for n in publications[k][0] :
            if (len(publications[k][0]) == 1 and n == "") :
                sys.stdout.write("No authors found")
            elif (count == len(publications[k][0]) - 1) :
                sys.stdout.write("and " + n + ".") 
            else :
                sys.stdout.write(n + ", ")
            count += 1
            
        print (f"\nYear published: {publications[k][1]}")
        print (f"Publication venue: {publications[k][2]}")
        print (f"Score: {publications[k][3]}")
        print ("---\n")

# export as a csv
def exportToCsv (fileName: str, _data : pd.DataFrame) -> None :
    if (_data.empty) :
        print ("Provide correct Data Frame")
        return

    if (not isinstance(fileName, str)) :
        print ("File name needs to be a string")
        return
        
    if (os.path.exists(f"./{fileName}.csv")) :
        warnings.warn ("This file already exists")
        return
    
    _data.to_csv(f"{fileName}.csv", index = False)

# export as a .txt file


The records should be manipulatable: at minimum you should be able to display sorted information according to descending year values, descending number of author values, the titles from A to Z, and finally by the impact of the papers.

In [ ]:
# sort the information: by years, by # of authors, titles (alphabetically)
# sort by impact of the papers
# sort by venue

# make a filter: i.e. display only the records from that year, implement AND and OR and NOT: display records from that year AND with this many authors
# ok maybe not
# focus on learning Pytorch then and maybe apply it in the second part

# maybe you can chain commands with the AND function, OR and NOT

Question 2: <br>
a. Selection of training set. Write code splitting the given dataset into the training and testing sets. The output of the code should be a dataset consisting of the rows of drone.csv that have been chosen. In the text part (part f) of your submission, please justify your choice.

In [ ]:
import random as rd
import matplotlib as mplt
from sklearn.feature_selection import r_regression, SelectKBest

droneData = pd.read_csv("drone.csv")

In [ ]:
def takeBootstrapSample (inBagProportion: float, df: pd.DataFrame) -> tuple :
    rowIndicies = []
    for i in range(1,droneData.shape[0] + 1) :
        rowIndicies.append(i)

    trainingSubsetRows = rd.sample(rowIndicies, round(len(rowIndicies) * inBagProportion))
    # converting to sets
    allRowsS = set (rowIndicies)
    trainingRowsS = set (trainingSubsetRows)
    validationSubsetRowsS = allRowsS - trainingRowsS
    # now get the actual rows from the dataframe
    trainingRowsBool = []
    validationRowsBool = []
    for i in rowIndicies :
        if i in trainingRowsS :
            trainingRowsBool.append(True)
        else :
            trainingRowsBool.append(False)

        if i in validationSubsetRowsS :
            validationRowsBool.append(True)
        else :
            validationRowsBool.append(False)

    trainingData = df.iloc[trainingRowsBool]
    validationData = df.iloc[validationRowsBool]

    return (trainingData, validationData)

bootstrapSample = takeBootstrapSample(0.75, droneData)


b. Feature selection (applied to the training set only). In this task you are requested to write code for testing of two feature selection approaches of your choice. The output of implementation of each approach should be the dataset with three features obtained from the output of part a) by removal of those columns that have not been selected as features.

In [ ]:
trainingDataset = bootstrapSample[0]

# this returns the number of NaNs for each column 
#print (trainingDataset.isna().sum())
# this returns a specific row that contains both NaNs: 3107
#print(trainingDataset[trainingDataset["load"].isna()])
# I can just remove it as it is a single row in more than 4000 observations
trainingDatasetRemNAs = trainingDataset.drop(3107)

# split the dataframe into the predictors and response
allPredictors = trainingDatasetRemNAs.loc[:, trainingDataset.columns != "warning"]
response = trainingDatasetRemNAs["warning"]

# convert to numpy array
allPredictorsNp = allPredictors.to_numpy()
responseNp = response.to_numpy()


# calculate the r-regression scores
scores = r_regression(allPredictorsNp, responseNp)

# select k best approach

# apparently that works with a data frame directly
selector = SelectKBest(r_regression, k = 3)
selectKBestRawOut = selector.fit_transform(allPredictors, response)

colIds = selector.get_support(indices = True)

selectKBestOutDf = trainingDataset.iloc[:, colIds]
print (selectKBestOutDf)

# select via L1 penalisation
#lasso = 



      current_a  current_filtered_a     scale
2      0.868000            0.530416  1.019714
5      0.954800            0.908717  1.024348
6      0.992000            0.921527  1.024756
7      0.917600            0.925939  1.024969
8      0.942400            0.926379  1.025161
...         ...                 ...       ...
5718  10.068799           11.119511  1.157143
5719   3.782000            9.517538  1.157143
5720   0.880400            4.778114  1.123704
5722   1.165600            1.881340  1.071099
5723   0.297600            1.135152  1.054939

[4294 rows x 3 columns]


Data binning (applied to the training set only) In this task you are requested to carry out data binning for the features of each data set obtained at the output of the previous part.

i. In particular, the values of each feature should be divided into 5 intervals with the smallest and largest values being the start and the end of the interval being partitioned.
ii. After that each value is to be replaced with the interval it belongs.
iii. Please implement two versions of binning for each data frame.
iv. In the first part just apply an ordinary binning. In the second part, the binning should be preceded by removal of outliers. The removal of outliers should be based on the interquartile range (IQR) method. Recall that, according to this method, a value is an outlier is it is smaller than the first quartile by more than 1.5IQR or greater than the third quartile by more than 1.5IQR.
v. Thus, the output of this step will consist of four datasets obtained by applying two ways of binning for each dataset obtained as the output of the previous step.
vi. In the text part (part f) of this task, please briefly discuss positive and negative effects of binning and removal of outliers in the context of ML.

In [ ]:
# could use CUT or to use MAP method of Pandas